In [10]:
!pip install -q faiss-cpu sentence-transformers

In [11]:
import os
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import faiss
from sentence_transformers import SentenceTransformer
import torch
import sklearn

In [12]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
torch.manual_seed(SEED)

Using device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
DATA_DIR = Path('/kaggle/input/datasets/kowie1/sdfghjk') 
documents = []
doc_names = []


for md_file in sorted(DATA_DIR.glob('*.md')):
    with open(md_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    if content:  # не пустой файл
        documents.append(content)
        doc_names.append(md_file.name)

print(f"Загружено документов: {len(documents)}")
print("Примеры названий файлов (первые 5):")
for name in doc_names[:5]:
    print(f"  - {name}")

print("Фрагмент первого документа (первые 500 символов):")
print(documents[0][:500] + "..." if len(documents[0]) > 500 else documents[0])

Загружено документов: 29
Примеры названий файлов (первые 5):
  - background-tasks.md
  - bigger-applications.md
  - body-fields.md
  - body-multiple-params.md
  - body-nested-models.md
Фрагмент первого документа (первые 500 символов):
# Background Tasks { #background-tasks }

You can define background tasks to be run *after* returning a response.

This is useful for operations that need to happen after a request, but that the client doesn't really have to be waiting for the operation to complete before receiving the response.

This includes, for example:

* Email notifications sent after performing an action:
    * As connecting to an email server and sending an email tends to be "slow" (several seconds), you can return the r...


Пояснение предметной области и пригодности для retrieval / mini-RAG

**Предметная область:**  
В папке `data/` находятся Markdown-файлы, содержащие справочные материалы по **FastAPI**.  
Документы структурированы: каждый файл посвящён отдельной подтеме.

**Почему по этой базе разумно строить retrieval и mini-RAG:**  
**Единая тематическая связность** — все документы относятся к одной области знаний, что позволяет формулировать осмысленные вопросы и ожидать релевантные ответы.  
**Содержательность** — тексты содержат определения, алгоритмы, примеры кода и сравнения, то есть информацию, на основе которой можно генерировать полезные ответы.  
**Умеренный объём** — 10–30 документов дают после чанкинга 30–150 фрагментов, что идеально для учебного retrieval без перегрузки.  


In [20]:
def chunk_text(text, chunk_size=300, overlap=50):
    """
    чанкинг по символам с перекрытием.
    """
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

CHUNK_SIZE = 300
OVERLAP = 50

all_chunks = []
chunk_metadata = []  # будем хранить (doc_index, chunk_index)

for doc_idx, doc in enumerate(documents):
    chunks = chunk_text(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            'doc_name': doc_names[doc_idx],
            'doc_index': doc_idx,
            'chunk_index': chunk_idx
        })

print(f"Всего чанков: {len(all_chunks)}")

print(f"Чанки первого документа ({doc_names[0]}):")
first_doc_chunks = [c for c, m in zip(all_chunks, chunk_metadata) if m['doc_name'] == doc_names[0]]
for i, chunk in enumerate(first_doc_chunks[:3]):
    print(f"--- Чанк {i} ---\n{chunk[:200]}...")

Всего чанков: 688
Чанки первого документа (background-tasks.md):
--- Чанк 0 ---
# Background Tasks { #background-tasks }

You can define background tasks to be run *after* returning a response.

This is useful for operations that need to happen after a request, but that the clien...
--- Чанк 1 ---
ion to complete before receiving the response.

This includes, for example:

* Email notifications sent after performing an action:
    * As connecting to an email server and sending an email tends to...
--- Чанк 2 ---
esponse right away and send the email notification in the background.
* Processing data:
    * For example, let's say you receive a file that must go through a slow process, you can return a response ...


**Выбранные значения:**
- `chunk_size = 300` символов
- `overlap = 50` символов

**Обоснование:**
- **Размер чанка 300 символов** (~50-70 слов) достаточен, чтобы вместить одно законченное определение или смысловой блок, но при этом достаточно мал для точного поиска.
- **Перекрытие 50 символов** (~15% от размера) помогает избежать разрыва ключевых фраз или предложений на границе чанков, повышая вероятность того, что релевантная информация не будет потеряна между фрагментами.

In [21]:
print("Создание эмбеддингов")
chunk_embeddings = model.encode(all_chunks, show_progress_bar=True)

#Размерность
dim = chunk_embeddings.shape[1]

#FAISS индекс (точный поиск по косинусному сходству через Inner Product)
index = faiss.IndexFlatIP(dim)
#Нормализуем эмбеддинги
faiss.normalize_L2(chunk_embeddings)
index.add(chunk_embeddings)

print(f"Индекс содержит {index.ntotal} векторов")

Создание эмбеддингов


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Индекс содержит 688 векторов


In [22]:
def search(query, k=3):
    """Возвращает список (doc_name, chunk_text, score) для top-k чанков"""
    q_emb = model.encode([query])
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        meta = chunk_metadata[idx]
        results.append({
            'doc_name': meta['doc_name'],
            'chunk_text': all_chunks[idx],
            'score': float(score)
        })
    return results

# Подставьте запросы, релевантные вашим документам
example_queries = [
    "what is FastAPI?",
    "how connect to API?",
    "how to work with requests"
]

print("Примеры поиска:\n")
for q in example_queries:
    print(f"Запрос: {q}")
    res = search(q, k=2)
    for i, r in enumerate(res):
        print(f"  {i+1}. [{r['doc_name']}] (score={r['score']:.3f})\n     {r['chunk_text'][:150]}...")
    print("-"*50)

Примеры поиска:

Запрос: what is FastAPI?
  1. [encoder.md] (score=0.677)
      is actually used by **FastAPI** internally to convert data. But it is useful in many other scenarios.

///...
  2. [index.md] (score=0.646)
     it in your editor is what really shows you the benefits of FastAPI, seeing how little code you have to write, all the type checks, autocompletion, etc...
--------------------------------------------------
Запрос: how connect to API?
  1. [first-steps.md] (score=0.519)
     ance" of the class `FastAPI`.

This will be the main point of interaction to create all your API.

### Step 3: create a *path operation* { #step-3-cre...
  2. [first-steps.md] (score=0.504)
     .0.1:8000](http://127.0.0.1:8000).

You will see the JSON response as:

```JSON
{"message": "Hello World"}
```

### Interactive API docs { #interactiv...
--------------------------------------------------
Запрос: how to work with requests
  1. [body.md] (score=0.537)
     # Request Body { #request-body }


In [23]:
eval_queries = [
    {"query": "How to run background tasks after returning a response in FastAPI?", "expected_doc": "background-tasks.md"},
    {"query": "How to structure a FastAPI application with multiple files and routers?", "expected_doc": "bigger-applications.md"},
    {"query": "How to declare a request body in FastAPI using Pydantic models?", "expected_doc": "body.md"},
    {"query": "How to define nested JSON objects and lists of submodels in request body with FastAPI?", "expected_doc": "body-nested-models.md"},
    {"query": "How to implement partial updates with PATCH and exclude_unset in FastAPI?", "expected_doc": "body-updates.md"},
    {"query": "How to raise HTTPException and handle errors globally in FastAPI?", "expected_doc": "handling-errors.md"},
    {"query": "How to declare path parameters with type hints and automatic validation in FastAPI?", "expected_doc": "path-params.md"},
    {"query": "How to use query parameters with default values and optional types in FastAPI?", "expected_doc": "query-params.md"},
    {"query": "How to enable CORS in FastAPI using CORSMiddleware to allow cross-origin requests?", "expected_doc": "cors.md"},
    {"query": "How to upload files in FastAPI using File, UploadFile, and handle multiple file uploads?", "expected_doc": "request-files.md"},
    {"query": "How to use Pydantic models to declare and validate form fields in FastAPI?", "expected_doc": "request-form-models.md"},
    {"query": "How to read and declare cookie parameters with Cookie in FastAPI?", "expected_doc": "cookie-params.md"}
]
K_VALUES = [1, 3, 5]

def evaluate_retrieval(queries, k_list):
    results = []
    search_k = max(max(k_list), 10)
    
    for item in queries:
        q = item['query']
        expected = item['expected_doc']
        retrieved = search(q, k=search_k)
        retrieved_docs = [r['doc_name'] for r in retrieved]

        rank = None
        for i, doc in enumerate(retrieved_docs):
            if doc == expected:
                rank = i + 1
                break

        row = {
            'query': q,
            'expected_source': expected,
            'retrieved_sources': ' | '.join(retrieved_docs[:max(k_list)]),
            'rank_of_first_relevant': rank if rank is not None else search_k + 1 
        }
        for k in k_list:
            row[f'hit@{k}'] = int(expected in retrieved_docs[:k])
            row[f'recall@{k}'] = int(expected in retrieved_docs[:k])
        results.append(row)
    return pd.DataFrame(results)

eval_df = evaluate_retrieval(eval_queries, K_VALUES)
eval_df['hit_at_k'] = eval_df['hit@3']
print("Результаты оценки на контрольных запросах: ")
display(eval_df)

Результаты оценки на контрольных запросах: 


,query,expected_source,retrieved_sources,rank_of_first_relevant,hit@1,recall@1,hit@3,recall@3,hit@5,recall@5,hit_at_k
0,How to run background tasks after returning a ...,background-tasks.md,background-tasks.md | background-tasks.md | ba...,1,1,1,1,1,1,1,1
1,How to structure a FastAPI application with mu...,bigger-applications.md,bigger-applications.md | bigger-applications.m...,1,1,1,1,1,1,1,1
2,How to declare a request body in FastAPI using...,body.md,body-multiple-params.md | body-multiple-params...,3,0,0,1,1,1,1,1
3,How to define nested JSON objects and lists of...,body-nested-models.md,body-nested-models.md | body-multiple-params.m...,1,1,1,1,1,1,1,1
4,How to implement partial updates with PATCH an...,body-updates.md,body-updates.md | body-updates.md | body-updat...,1,1,1,1,1,1,1,1
5,How to raise HTTPException and handle errors g...,handling-errors.md,handling-errors.md | handling-errors.md | hand...,1,1,1,1,1,1,1,1
6,How to declare path parameters with type hints...,path-params.md,query-param-models.md | header-param-models.md...,4,0,0,0,0,1,1,0
7,How to use query parameters with default value...,query-params.md,query-params-str-validations.md | query-params...,10,0,0,0,0,0,0,0
8,How to enable CORS in FastAPI using CORSMiddle...,cors.md,cors.md | cors.md | cors.md | cors.md | cors.md,1,1,1,1,1,1,1,1
9,"How to upload files in FastAPI using File, Upl...",request-files.md,request-files.md | request-files.md | bigger-a...,1,1,1,1,1,1,1,1


In [24]:
print("СРЕДНИЕ МЕТРИКИ: ")

recall_scores = {}
for k in K_VALUES:
    hit = eval_df[f'hit@{k}'].mean()
    recall = eval_df[f'recall@{k}'].mean()
    recall_scores[k] = recall 
    print(f"Hit@{k}  : {hit:.3f} ")
    print(f"Recall@{k}: {recall:.3f} ")

СРЕДНИЕ МЕТРИКИ: 
Hit@1  : 0.750 
Recall@1: 0.750 
Hit@3  : 0.833 
Recall@3: 0.833 
Hit@5  : 0.917 
Recall@5: 0.917 


In [25]:
# === Эксперимент: сравнение top_k = 3 и top_k = 5 ===

print("ЭКСПЕРИМЕНТ: сравнение Hit при k=3 и k=5")
k1, k2 = 3, 5
hits_k1 = []
hits_k2 = []
for item in eval_queries:
    q = item['query']
    expected = item['expected_doc']
    res = search(q, k=max(k1, k2))
    docs = [r['doc_name'] for r in res]
    hits_k1.append(int(expected in docs[:k1]))
    hits_k2.append(int(expected in docs[:k2]))

print(f"Средний Hit@{k1}: {np.mean(hits_k1):.3f}")
print(f"Средний Hit@{k2}: {np.mean(hits_k2):.3f}")
print("Вывод: увеличение k повышает вероятность найти релевантный документ,")
print("но увеличивает объём нерелевантного контекста.")

ЭКСПЕРИМЕНТ: сравнение Hit при k=3 и k=5
Средний Hit@3: 0.833
Средний Hit@5: 0.917
Вывод: увеличение k повышает вероятность найти релевантный документ,
но увеличивает объём нерелевантного контекста.


In [26]:
# === 2.3.7. Обновление базы знаний и переиндексация ===

new_docs = [
    ("responses.md", 
     "FastAPI supports many response types: JSONResponse, HTMLResponse, PlainTextResponse, FileResponse, and RedirectResponse. Use them to return different content formats."),

    ("status-codes.md", 
     "HTTP status codes indicate the result of a request. FastAPI provides the `status` module with constants like `status.HTTP_200_OK` or `status.HTTP_404_NOT_FOUND` for better readability.")
]

# Расширяем списки
documents.extend([content for _, content in new_docs])
doc_names.extend([name for name, _ in new_docs])

# Повторный чанкинг
all_chunks_new = []
chunk_metadata_new = []
for doc_idx, doc in enumerate(documents):
    chunks = chunk_text(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks_new.append(chunk)
        chunk_metadata_new.append({
            'doc_name': doc_names[doc_idx],
            'doc_index': doc_idx,
            'chunk_index': chunk_idx
        })

print(f"После обновления: документов = {len(documents)}, чанков = {len(all_chunks_new)}")

# Новые эмбеддинги и индекс
new_embeddings = model.encode(all_chunks_new, show_progress_bar=True)
faiss.normalize_L2(new_embeddings)
index_new = faiss.IndexFlatIP(dim)
index_new.add(new_embeddings)

# Функция поиска по новому индексу
def search_new(query, k=3):
    q_emb = model.encode([query])
    faiss.normalize_L2(q_emb)
    scores, indices = index_new.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        meta = chunk_metadata_new[idx]
        results.append({
            'doc_name': meta['doc_name'],
            'chunk_text': all_chunks_new[idx],
            'score': float(score)
        })
    return results

print("Индекс обновлён.")

После обновления: документов = 31, чанков = 690


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Индекс обновлён.


In [27]:
# === Сравнение до и после обновления ===

# Запросы, на которых ожидается влияние новых документов
update_test_queries = [
    {"query": "What response types are available in FastAPI?"},
    {"query": "How to use HTTP status codes in FastAPI?"},
    {"query": "What HTTP methods decorators does FastAPI provide?"}
]

compare_data = []
for item in update_test_queries:
    q = item['query']
    before = search(q, k=3)
    after = search_new(q, k=3)
    before_docs = [r['doc_name'] for r in before]
    after_docs = [r['doc_name'] for r in after]
    changed = before_docs != after_docs
    compare_data.append({
        'query': q,
        'before_retrieved_sources': ' | '.join(before_docs),
        'after_retrieved_sources': ' | '.join(after_docs),
        'changed': changed
    })

compare_df = pd.DataFrame(compare_data)
print("Сравнение результатов до и после обновления:")
display(compare_df)

Сравнение результатов до и после обновления:


,query,before_retrieved_sources,after_retrieved_sources,changed
0,What response types are available in FastAPI?,path-operation-configuration.md | handling-err...,responses.md | path-operation-configuration.md...,True
1,How to use HTTP status codes in FastAPI?,path-params.md | path-params.md | first-steps.md,status-codes.md | responses.md | path-params.md,True
2,What HTTP methods decorators does FastAPI prov...,path-params.md | first-steps.md | first-steps.md,path-params.md | first-steps.md | responses.md,True


In [28]:
# === Mini-RAG ===

def mini_rag(question, k=3):
    """
    Простой RAG-конвейер:
    - получает запрос
    - извлекает top-k чанков из обновлённого индекса
    - формирует ответ на основе контекста (шаблонный)
    - возвращает ответ и источники
    """
    retrieved = search_new(question, k=k)
    if not retrieved:
        answer = "В базе знаний не найдено релевантной информации."
        sources = []
        return answer, sources, retrieved
    
    context = "\n\n".join([f"[{r['doc_name']}]: {r['chunk_text']}" for r in retrieved])
    answer = f"Ответ основан на следующих фрагментах:\n\n"
    answer += f"Основной источник: {retrieved[0]['doc_name']}\n"
    answer += f"Контекст: {retrieved[0]['chunk_text'][:300]}..."
    
    sources = list(set([r['doc_name'] for r in retrieved]))
    return answer, sources, retrieved

# Примеры работы mini-RAG
rag_examples = []
test_questions = [
    "How to run background tasks in FastAPI?",
    "How to declare a request body with Pydantic model?",
    "How to use query parameters with default values?"
]

print("=== ПРИМЕРЫ РАБОТЫ MINI-RAG ===")
for q in test_questions:
    ans, srcs, _ = mini_rag(q)
    rag_examples.append({
        'question': q,
        'answer': ans,
        'retrieved_sources': ' | '.join(srcs)
    })
    print(f"Вопрос: {q}")
    print(f"Ответ: {ans[:250]}...")
    print(f"Источники: {srcs}\n")

=== ПРИМЕРЫ РАБОТЫ MINI-RAG ===
Вопрос: How to run background tasks in FastAPI?
Ответ: Ответ основан на следующих фрагментах:

Основной источник: background-tasks.md
Контекст:  it's then possible to use it as a *path operation function* parameter and have **FastAPI** handle the rest for you, just like when using the `Request` object di...
Источники: ['background-tasks.md']

Вопрос: How to declare a request body with Pydantic model?
Ответ: Ответ основан на следующих фрагментах:

Основной источник: body.md
Контекст: clare a **request** body, you use [Pydantic](https://docs.pydantic.dev/) models with all their power and benefits.

/// info

To send data, you should use one of: `POST` (th...
Источники: ['body-fields.md', 'body.md']

Вопрос: How to use query parameters with default values?
Ответ: Ответ основан на следующих фрагментах:

Основной источник: query-params.md
Контекст: t want to add a specific value but just make it optional, set the default as `None`.

But when you want to make 


# Пример 1
Запрос: 'How to deploy FastAPI to AWS Lambda?'
Найденные документы: first-steps.md, index.md
Проблема: в базе нет информации про деплой на AWS Lambda, только базовые шаги
Результат: retrieval находит общие статьи про деплой, но не отвечает на конкретный вопрос

# Пример 2
Запрос: 'How to integrate FastAPI with GraphQL?'
Найденные документы: first-steps.md, path-operation-configuration.md
Проблема: тема GraphQL отсутствует в документации FastAPI (только упоминание в first-steps)
Результат: ответ не содержит нужной информации, retrieval выдаёт нерелевантные документы.

# Пример 3
Запрос: 'What is the difference between Depends and BackgroundTasks?'
Найденные документы: background-tasks.md, dependencies.md (после обновления)
Проблема: хотя оба документа есть, в них нет прямого сравнения. Контекст содержит отдельные описания, но не объясняет разницу.
Результат: mini-RAG выдаёт фрагменты обоих документов, но не синтезирует сравнение.

In [29]:
# === Сохранение артефактов в соответствии с ТЗ ===

from pathlib import Path
import shutil

# Создаём структуру homeworks/HW14/artifacts в /kaggle/working
WORK_DIR = Path('/kaggle/working')
HW14_DIR = WORK_DIR / 'homeworks' / 'HW14'
ARTIFACTS_DIR = HW14_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. retrieval_eval.csv
eval_df.to_csv(ARTIFACTS_DIR / 'retrieval_eval.csv', index=False)

# 2. rag_examples.csv
rag_df = pd.DataFrame(rag_examples)
rag_df.to_csv(ARTIFACTS_DIR / 'rag_examples.csv', index=False)

# 3. retrieval_before_after_update.csv
compare_df.to_csv(ARTIFACTS_DIR / 'retrieval_before_after_update.csv', index=False)

print("Артефакты сохранены в:", ARTIFACTS_DIR)
!ls -la {ARTIFACTS_DIR}

Артефакты сохранены в: /kaggle/working/homeworks/HW14/artifacts
total 20
drwxr-xr-x 2 root root 4096 Apr 15 08:35 .
drwxr-xr-x 3 root root 4096 Apr 15 08:35 ..
-rw-r--r-- 1 root root 1609 Apr 15 08:35 rag_examples.csv
-rw-r--r-- 1 root root  551 Apr 15 08:35 retrieval_before_after_update.csv
-rw-r--r-- 1 root root 2747 Apr 15 08:35 retrieval_eval.csv
